# How effectively are Queensland public dental clinics 
# managing general type patient demand across catchments?

In [72]:
# Import Data
from pathlib import Path
import pandas as pd

# Current working directory (notebooks folder)
BASE_DIR = Path.cwd()

# Move up one level to project root
PROJECT_ROOT = BASE_DIR.parent

# Relative paths

PATHS = {
    'quarterly_totals' :        PROJECT_ROOT / "data" / "processed" / "EDA" / "quarterly_totals.csv",
    'average_visit' :           PROJECT_ROOT / "data" / "processed" / "EDA" / "average_visit.csv",
    'quarterly_capacity' :      PROJECT_ROOT / "data" / "processed" / "EDA" / "quarterly_capacity.csv",
    'catchment_capacity' :      PROJECT_ROOT / "data" / "processed" / "EDA" / "catchment_capacity.csv",
    'clinic_backlog' :          PROJECT_ROOT / "data" / "processed" / "EDA" / "clinic_backlog.csv",
}

# Descriptions
        # quarter_visit
        # quarter_visit_pct      -- Quarterly percentage of total waiting per type of visit
        # avg_clinic_capacity      -- total waiting and treated, per clinic & catchment

df_quarterly_totals    = pd.read_csv(PATHS['quarterly_totals'])
df_quarterly_capacity  = pd.read_csv(PATHS['quarterly_capacity'])
df_avg_visit_pct       = pd.read_csv(PATHS['average_visit'])
# df_catchment_avg       = pd.read_csv(PATHS['avg_clinic_capacity'])

## Overall Patient Waitlist and Treatment Volume by Quarter
Trends in both patients waiting in queues and treated by clinics each quarter. \
Complete missing quarters are highlighted for clarity.  

In [114]:
# visualise quarterly patients treated and waiting

##Missing Months:
##                2020 --  1, 4-6, 12  - Q1 and Q4 quarter are partial. 
##                2021 --  7-12 - Q3 & Q4  
##                2022 --   1-6 - Q1 & Q2 
##                2023 --   1-3 - Q1

import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(specs=[[{"secondary_y": True}]])


# Add traces
fig.add_trace(go.Scatter(
    x=df_quarterly_totals['quarter_start'], 
    y=df_quarterly_totals['waiting'],
    mode='lines',
    name='Patients Waiting',
    text=[''] * (len(df_quarterly_totals) - 1) + ['Waited'],  # label only on last point
    textposition='middle right',
    textfont=dict(color='rgba(255, 100, 100, 0.9)', size=11),
    fill='tonexty',                         #fills to next
    fillcolor='rgba(100, 149, 237, 0.1)',  # light blue fill
    line=dict(color='rgba(100, 149, 237, 0.4)', width=1.5),
    showlegend=False,                     # ← hide from legend box
), secondary_y=False)

fig.add_trace(go.Scatter(
    x=df_quarterly_totals['quarter_start'],
    y=df_quarterly_totals['treated'],
    text=[''] * (len(df_quarterly_totals) - 1) + ['Treated'],
    textposition='middle right',
    textfont=dict(color='rgba(255, 100, 100, 0.9)', size=11),
    mode='lines',
    name='Patients Treated',
    fill='tozeroy',                        # fills down to zero
    fillcolor='rgba(37, 99, 235, 0.3)', # medium blue fill
    line=dict(color='rgba(59, 130, 246, 0.8)', width=1.5),
    showlegend=False
), secondary_y=True)

fig.update_yaxes(
    range=[0, df_quarterly_totals['waiting'].max()*1.1],
    secondary_y=True
)
fig.update_yaxes(
    range=[0, df_quarterly_totals['waiting'].max()*1.1],
    secondary_y=False
)

fig.for_each_xaxis(lambda x: x.update(showgrid=False)) #https://stackoverflow.com/questions/67459925/plotly-set-showgrid-false-for-all-subplots
fig.for_each_yaxis(lambda x: x.update(showgrid=False))

# Connect the lines across missing data points, 
# so the line will continue through the shaded areas without breaks.
fig.update_traces(connectgaps=True) 

# Shade each missing monthly period

fig.add_vrect(x0='2019-09-01', x1='2019-12-31', 
              fillcolor='lightcoral', opacity=0.1, layer='below', line_width=0)
fig.add_vrect(x0='2020-04-01', x1='2020-06-30', 
              fillcolor='lightcoral', opacity=0.1, layer='below', line_width=0)
# 2021: Q3-Q4 missing
fig.add_vrect(x0='2021-07-01', x1='2021-12-31',
              fillcolor='lightcoral', opacity=0.1, layer='below', line_width=0,
              annotation_text='Missing Data',  
              annotation=dict(xref='paper', 
                              yref='paper',
                              xshift=15, #adds margin
                              font_size=12, 
                              font_color='salmon'))

# 2022: Q1-Q2 missing  
fig.add_vrect(x0='2022-01-01', x1='2022-06-30',
              fillcolor='lightcoral', opacity=0.1, layer='below', line_width=0)

# 2023: Q1 missing
fig.add_vrect(x0='2023-01-01', x1='2023-03-31',
              fillcolor='lightcoral', opacity=0.1, layer='below', line_width=0)
fig.add_vline(x='2023-06-01', line_width=2, line_dash="dot", line_color="slategray",)
fig.add_annotation(x=pd.to_datetime('2023-06-01'),
                   y=-0.01,  # position below x-axis
                   ay=20,  # offset in pixels
                   ax=40,  # offset in pixels
                   axref='pixel',
                   ayref='pixel',
                   yref='paper',
                   text="Quarterly Collection Starts",
                   xanchor='left',
                   yanchor='top',
                   showarrow=True)

# annotations to name the lines on the right side
fig.add_annotation(
    xref="paper", yref="paper",
    x=1.01, y=0.05,
    text="<span style='color:rgba(59, 130, 246, 0.8); font-weight:bold;'>Treated</span>",
    showarrow=False
)
fig.add_annotation( 
    xref="paper", yref="paper",
    x=1.01, y=0.95,
    text="<span style='color:rgba(37, 99, 235, 0.5); font-weight:bold;'>Waiting</span>",
    showarrow=False
)

fig.update_layout(
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="right", x=0.805),
    margin=dict(
        t=50,  # top
        b=80,  # bottom  ← your problem
        l=50,  # left
        r=80,  # right
    ),
    plot_bgcolor='white',
    title='Quarterly Patient Waiting List Volume',
    xaxis_title='',
    yaxis_title='',
    hovermode="x",
)

fig.update_yaxes(visible=False, secondary_y=True) #hides secondary axis
#fig.update_traces(mode="markers+lines", hovertemplate=None) #adds marks to line plots
fig.show()


In [69]:
#increase measured + description
wait_increase = (df_quarterly_totals.loc[df_quarterly_totals['quarter_start'] == '2025-04-01', 'waiting'].values[0] - df_quarterly_totals.loc[df_quarterly_totals['quarter_start'] == '2024-04-01', 'waiting'].values[0])
general = df_avg_visit_pct.loc[df_avg_visit_pct['visit_type'] == 'General', 'pct_of_list'].values[0]
priority3 = df_avg_visit_pct.loc[df_avg_visit_pct['visit_type'] == 'Priority 3', 'pct_of_list'].values[0]
other = (
            df_avg_visit_pct.loc[df_avg_visit_pct['visit_type'] == 'Priority 1', 'pct_of_list'].values[0] +
            df_avg_visit_pct.loc[df_avg_visit_pct['visit_type'] == 'Priority 2', 'pct_of_list'].values[0] +
            df_avg_visit_pct.loc[df_avg_visit_pct['visit_type'] == 'General Anaesthetic Category 1', 'pct_of_list'].values[0]+
            df_avg_visit_pct.loc[df_avg_visit_pct['visit_type'] == 'General Anaesthetic Category 2', 'pct_of_list'].values[0]+
            df_avg_visit_pct.loc[df_avg_visit_pct['visit_type'] == 'General Anaesthetic Category 3', 'pct_of_list'].values[0]+
            df_avg_visit_pct.loc[df_avg_visit_pct['visit_type'] == 'Clinical Assessment', 'pct_of_list'].values[0]
)
print(f"Despite the more recent consistant data collections, the volume of treated patients have not improved. \nWaiting list gap is growing. From April 2023 to April 2025 has steadily increased by {wait_increase:,} patients.\n")
print(f"The distribution of appointment types on average come to General leading with {general}%, \nfollowed by {priority3}% in Priority 3, and the rest of the appointments by {other}%")

Despite the more recent consistant data collections, the volume of treated patients have not improved. 
Waiting list gap is growing. From April 2023 to April 2025 has steadily increased by 16,772 patients.

The distribution of appointment types on average come to General leading with 88.16%, 
followed by 6.37% in Priority 3, and the rest of the appointments by 5.48%


*Why focus only on General Appointments?*
> Their large scale can influence overall system performance, making them a \
> plausible anchor for this analysis. While other appointment types \
> are out of focus here, they may surface useful findings in future analyses.

## Is treatment keeping pace with demand?

To better understand this question, we will use the treatment rate to determine a \
waitlist's expected completion time in months, then compare that figure to the \
desired wait time for a General appointment.

_What is the treatment rate?_
> It is the ratio of treated to waiting patients per quarter. 
>
> General appointments have a desired wait time of 24months. Using a treatment rate\
> we can gauge the capacity of a waitlist against that benchmark. When a quarter\
> requires more than 24 months to clear its waitlist volume, it is operating over capacity.

In [78]:

import plotly.express as px
colours = ['blue' if v < 24 else 'lightcoral' for v in df_quarterly_capacity['months_to_treat']]

#bar plot
fig = px.bar(
    df_quarterly_capacity,
    x='quarter_start',
    y='months_to_treat',
    title='Months to Clear Waitlist at Current Quarterly Treatment Rate',
    labels={
        'months_to_treat': 'Months',
        'quarter_start': 'Quarter Period',}
)

fig.update_traces(
    marker_color=colours,
    marker_opacity=0.6
)

#dashed line at 24 month threshold
fig.add_hline(
    y=24, 
    line_dash="dash", 
    line_color="salmon"
)
fig.add_annotation(
                   y=24,
                   yref='y',
                   xref='paper',
                   x=.8, # place annotation at 75% of x-axis
                   text="24 Month Threshold",
                   showarrow=False,
                   xanchor='left',
                   yanchor='bottom',
                   font=dict(size=12, color="lightcoral"))
#shade area above threshold

#fig.add_hrect(
#    y0=24, 
#    y1=df_quarterly_capacity['months_to_treat'].max() * 1.1, 
#    fillcolor="lightcoral", 
#    opacity=0.2, 
#    layer="below", 
#    line_width=0
#)

#change background to white
fig.update_layout(plot_bgcolor='white')

fig.show()